# Tier-a Phase 1 — Kaggle pipeline test + profiling

Runs the three steps of `docs/tier_a_runbook.md` Phase 1 on Kaggle's free
T4×2: **1a** CUDA smoke, **1b** rollout profile, **1c** short end-to-end
pipeline run. Objective is *pipeline validation + profiling*, not strength
(Kaggle's ~4 vCPU starve the CPU-bound rollout; that's expected).

## One-time setup (manual, in the Kaggle UI)
1. Create a Kaggle account and **verify your phone number**
   (Settings → Phone verification) — this unlocks GPU accelerators.
2. In this notebook's editor: **Settings → Accelerator → GPU T4 x2**,
   **Settings → Internet → On** (needed for the git clone).
3. Quota: ~30 GPU-h/week, sessions capped at 9–12 h. Interactive sessions
   die when the tab disconnects too long; **Save Version → Save & Run All
   (Commit)** runs in the background and survives closing the tab.
4. Do **NOT** `pip install -r requirements.txt` — it pins `torch-directml`
   (Windows-only). Everything needed (torch, numpy) is preinstalled.

Artifacts land in `/kaggle/working/` and are downloadable from the
notebook's **Output** tab after a committed run.

In [ ]:
# Environment check: GPU visible, torch sees CUDA, how many vCPUs.
!nvidia-smi
import os, torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(),
      '| devices:', torch.cuda.device_count())
print('vCPUs:', os.cpu_count())
assert torch.cuda.is_available(), 'No CUDA — set Accelerator to GPU T4 x2'

In [ ]:
# Fresh clone each run (the repo is self-contained: pinned 1.18.4 scrapes,
# the wesnoth_src runtime WML subset, and tier_a_5m.pt are all committed).
# Clone OUTSIDE /kaggle/working: everything under working/ becomes notebook
# output, and shipping the whole repo back (~80MB with .git) drowns the
# actual artifacts.
!rm -rf /tmp/Wesnoth-AI
!git clone --depth 1 https://github.com/momom2/Wesnoth-AI.git /tmp/Wesnoth-AI
%cd /tmp/Wesnoth-AI

In [ ]:
# Sanity: runtime data + warm-start checkpoint present, arch as expected.
from pathlib import Path
for p in ['wesnoth_src/data/multiplayer/factions',
          'wesnoth_src/data/multiplayer/scenarios',
          'wesnoth_src/data/multiplayer/maps',
          'wesnoth_src/data/add-ons/Mini_Maps_Collection',
          'unit_stats.json', 'terrain_db.json',
          'training/checkpoints/tier_a_5m.pt']:
    assert Path(p).exists(), f'MISSING: {p}'
    print('ok', p)
import torch
ck = torch.load('training/checkpoints/tier_a_5m.pt', map_location='cpu', weights_only=False)
n = sum(v.numel() for v in ck['model_state'].values())
print('arch:', ck['arch'], '| params: %.2fM' % (n / 1e6),
      '| decision_step:', ck['decision_step'])

## 1a — REQUIRED CUDA smoke (runbook §Phase 1a)
Pass criteria: exit 0, no device-mismatch errors, log shows
`mcts_batch_size = 16 (... device=cuda)` and `train_batch_size = 8` with
NO thread-cap line, and the checkpoint saves.

In [ ]:
!python tools/sim_self_play.py --device cuda \
  --mcts --mcts-sims 8 --iterations 2 --games-per-iter 2 --max-turns 12 \
  --mini-ratio 1.0 --ladder-ratio 0.0 --replay-buffer --replay-updates 2 --replay-minibatch 16 \
  --replay-min-size 1 --train-batch-size 8 \
  --d-model 256 --num-layers 6 --num-heads 8 --d-ff 1024 \
  --checkpoint-in training/checkpoints/tier_a_5m.pt \
  --checkpoint-out /kaggle/working/gpu_smoke.pt --log-level INFO
!ls -la /kaggle/working/gpu_smoke.pt*

## 1b — Profile the rollout (runbook §Phase 1b)
**Post-patch re-profile.** The sampler-on-CPU split + B2 batch reads
(`docs/gpu_perf_patches.md` #1/#2) were applied 2026-07-02. Baseline to
beat (T4, pre-patch, 2026-07-02): forward 43.8% at 6.7ms/leaf,
enumerate 23.0%, encode 23.8%, 2.7 actions/s GPU-serialized. Expect
`forward`'s us/call to drop (the enumeration D2H syncs were billed
there) and overall actions/s to rise. This run is also the CUDA
correctness validation for the patch — a missed tensor field would
crash with a device mismatch here, which CPU tests cannot catch.

In [ ]:
!python tools/profile_rollout.py --device cuda \
  --checkpoint-in training/checkpoints/tier_a_5m.pt \
  --d-model 256 --num-layers 6 --num-heads 8 --d-ff 1024 \
  --games 2 --mcts-sims 32 --mini-ratio 0.5 \
  --save-json /kaggle/working/profile.json --log-level INFO

## 1c — Short end-to-end pipeline run (runbook §Phase 1c)
~20 iterations: confirms the loop, replay buffer, checkpoint/resume and
WHR/Elo logging on CUDA. First launch of the fresh campaign → keeps
`--reset-decision-step`. **Gate to Phase 2:** smoke green, GPU fed after
perf fixes, held-out value loss trending down.

**T4 memory note (learned from the 2026-07-02 run):** the runbook's
Phase 2 values (`--replay-minibatch 128 --train-batch-size 128`) OOM'd
the 15GB T4 at iteration 1 (12.7GB allocated, +818MB refused). Here we
use minibatch 64 / chunk 32; `expandable_segments` fights allocator
fragmentation. The 24GB 4090 in Phase 2 keeps the original 128/128.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python tools/sim_self_play.py --device cuda \
  --mcts --mcts-sims 32 --iterations 20 --games-per-iter 8 --max-turns 24 \
  --mini-ratio 0.4 --drill-ratio 0.2 --ladder-ratio 0.4 \
  --replay-buffer --replay-updates 16 --value-coef 1.0 \
  --replay-minibatch 64 --replay-capacity 6000 --train-batch-size 32 \
  --d-model 256 --num-layers 6 --num-heads 8 --d-ff 1024 \
  --holdout-size 256 \
  --reset-decision-step \
  --checkpoint-in training/checkpoints/tier_a_5m.pt \
  --checkpoint-out /kaggle/working/tier_a_5m.pt \
  --save-every 2 --log-level INFO
# --holdout-size: watch the per-iter `holdout value CE=` line -- that's
# the generalization metric (train value loss is measured on replay
# samples the net already fit). No abort tripwire here: a 20-iter
# warm-up run is EXPECTED to draw everything; the tripwire is a
# Phase 2 guard (see the runbook).

In [ ]:
# Collect artifacts for download (Output tab): this run's history rows
# land in trainer_history_local.csv (the numbered CSVs are committed
# cluster-era curves -- skip them).
!cp -v training/logs/trainer_history_local.csv /kaggle/working/ 2>/dev/null || true
!ls -la /kaggle/working/